# Create Study Folder In Google Drive

This Colab notebook publishes a cleaned local study folder into a Google Drive study-folder template.

It can:

- authenticate with Google Drive and Google Sheets;
- prompt you to upload a zipped local study folder;
- initialize a Google Drive study folder from a template;
- fill `IRB-meta`, the study overview, cleaned data workbooks, and data maps;
- write a local workflow log back into the uploaded study folder.

Before starting, zip your local study folder. Example:

```bash
zip -r 53879-62882-OCD-TMS.zip studies/53879-62882-OCD-TMS
```

By default, the notebook refreshes workflow scripts from GitHub so you do not accidentally run stale scripts from an uploaded zip. If you want to use scripts bundled inside your zip instead, set `REFRESH_SCRIPTS_FROM_REPO = False` in the configuration cell.

In [ ]:
# @title 1. Connect To Google Drive {"run":"auto","display-mode":"form"}
!pip install -q openpyxl google-auth google-api-python-client

from google.colab import auth
auth.authenticate_user()

import google.auth
from google.auth.transport.requests import Request

creds, project_id = google.auth.default(
    scopes=[
        "https://www.googleapis.com/auth/drive",
        "https://www.googleapis.com/auth/spreadsheets",
    ]
)
creds.refresh(Request())
access_token = creds.token

print("Authenticated with Google Drive + Google Sheets scopes.")

In [ ]:
# @title 2. Configure Study Upload {"display-mode":"form"}

# Required study fields.
STUDY_NAME = "OCD-TMS"  # @param {type:"string"}
IRB = "53879-62882"  # @param {type:"string"}

# Google Drive template and destination.
TEMPLATE_FOLDER_URL = "https://drive.google.com/drive/folders/1OLGHgxPg9UsBDbOH6vgSjiS6dUW0lBCF?usp=sharing"  # @param {type:"string"}
DESTINATION_PARENT_FOLDER = "root"  # @param {type:"string"}

# Workflow stage. Use all for normal publishing.
STAGE = "all"  # @param ["all", "initialize", "upload", "data-map"]
EXISTING_FILE_POLICY = "update-or-create"  # @param ["update-or-create", "skip", "duplicate", "replace", "fail"]

# Workflow scripts source. Keep refresh enabled to avoid stale scripts from an uploaded zip.
REPO_URL = "https://github.com/tracyhann/bsl-data-workflows.git"  # @param {type:"string"}
REFRESH_SCRIPTS_FROM_REPO = True  # @param {type:"boolean"}

# Optional overrides. Leave blank to use template defaults.
INITIALIZED_FOLDER_ID = ""  # @param {type:"string"}
OVERVIEW_DESTINATION = ""  # @param {type:"string"}
CLEANED_DATA_FOLDER_ID = ""  # @param {type:"string"}
TEMPLATES_FOLDER_ID = ""  # @param {type:"string"}
DATA_MAP_DESTINATION = ""  # @param {type:"string"}

print("Configuration loaded.")

In [ ]:
# @title 3. Upload Local Study Folder Zip {"display-mode":"form"}
from google.colab import files
from pathlib import Path
import zipfile
import shutil
import os

os.chdir("/content")
WORKSPACE = Path("/content/BSLDB")
UPLOAD_DIR = Path("/content/uploads")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)
WORKSPACE.mkdir(parents=True, exist_ok=True)

print("Choose a .zip containing either:")
print("  1. the full repo, including scripts/ and studies/...")
print("  2. only the local study folder, e.g. studies/53879-62882-OCD-TMS")

uploaded = files.upload(target_dir=str(UPLOAD_DIR))
if not uploaded:
    raise RuntimeError("No zip file uploaded.")

zip_name = next(iter(uploaded.keys()))
zip_path = UPLOAD_DIR / Path(zip_name).name
if zip_path.suffix.lower() != ".zip":
    raise ValueError(f"Expected a .zip file, got: {zip_path}")

with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(WORKSPACE)

print(f"Extracted {zip_path.name} into {WORKSPACE}")
print("Top-level extracted contents:")
for item in sorted(WORKSPACE.iterdir()):
    print(" -", item.relative_to(WORKSPACE))

In [ ]:
# @title 4. Prepare Workflow Scripts {"display-mode":"form"}
from pathlib import Path
import shutil
import subprocess
import os

os.chdir(WORKSPACE)

def find_first(path_root, relative_parts):
    for candidate in path_root.rglob(relative_parts[-1]):
        if candidate.is_dir() and candidate.parts[-len(relative_parts):] == tuple(relative_parts):
            return candidate
    return None

scripts_dir = WORKSPACE / "scripts"
repo_url = REPO_URL.strip()
if repo_url and REFRESH_SCRIPTS_FROM_REPO:
    repo_dir = WORKSPACE / "workflow_repo"
    if repo_dir.exists():
        shutil.rmtree(repo_dir)
    if scripts_dir.exists():
        shutil.rmtree(scripts_dir)
    subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)
    shutil.copytree(repo_dir / "scripts", scripts_dir)

if not scripts_dir.exists():
    nested_scripts = find_first(WORKSPACE, ["scripts"])
    if nested_scripts and nested_scripts != scripts_dir:
        shutil.copytree(nested_scripts, scripts_dir)

if not scripts_dir.exists():
    if not repo_url:
        raise RuntimeError(
            "No scripts/ folder found in the uploaded zip. Set REPO_URL in cell 2, "
            "or upload a zip containing the full repo."
        )
    repo_dir = WORKSPACE / "workflow_repo"
    if repo_dir.exists():
        shutil.rmtree(repo_dir)
    subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)
    shutil.copytree(repo_dir / "scripts", scripts_dir)

run_script = scripts_dir / "workflows" / "create_study_folder_gdrive" / "run.py"
if not run_script.exists():
    raise FileNotFoundError(f"Missing workflow script: {run_script}")

print(f"Workflow script ready: {run_script}")

In [ ]:
# @title 5. Detect Or Choose Local Study Folder {"display-mode":"form"}
from pathlib import Path

# Leave blank to auto-detect the first folder containing data/cleaned.
LOCAL_STUDY_FOLDER = ""  # @param {type:"string"}

def candidate_study_folders(root):
    candidates = []
    for path in root.rglob("data"):
        if path.is_dir() and (path / "cleaned").exists():
            candidates.append(path.parent)
    return sorted(candidates, key=lambda p: len(p.parts))

if LOCAL_STUDY_FOLDER.strip():
    study_folder = Path(LOCAL_STUDY_FOLDER.strip())
    if not study_folder.is_absolute():
        study_folder = WORKSPACE / study_folder
else:
    candidates = candidate_study_folders(WORKSPACE)
    if not candidates:
        raise RuntimeError("Could not find a study folder with data/cleaned. Set LOCAL_STUDY_FOLDER manually.")
    study_folder = candidates[0]

if not study_folder.exists():
    raise FileNotFoundError(study_folder)

print(f"Using local study folder: {study_folder}")
print("Relative path:", study_folder.relative_to(WORKSPACE) if WORKSPACE in study_folder.parents else study_folder)

In [ ]:
# @title 6. Create Google Drive Study Folder + Upload Data {"display-mode":"form"}
import subprocess

cmd = [
    "python3", "scripts/workflows/create_study_folder_gdrive/run.py",
    "--stage", STAGE,
    "--study-folder", str(study_folder),
    "--study-name", STUDY_NAME,
    "--irb", IRB,
    "--access-token", access_token,
]

if TEMPLATE_FOLDER_URL.strip():
    cmd.extend(["--template", TEMPLATE_FOLDER_URL.strip()])
if DESTINATION_PARENT_FOLDER.strip():
    cmd.extend(["--destination", DESTINATION_PARENT_FOLDER.strip()])
if INITIALIZED_FOLDER_ID.strip():
    cmd.extend(["--initialized-folder-id", INITIALIZED_FOLDER_ID.strip()])
if OVERVIEW_DESTINATION.strip():
    cmd.extend(["--overview-destination", OVERVIEW_DESTINATION.strip()])
if CLEANED_DATA_FOLDER_ID.strip():
    cmd.extend(["--cleaned-data-folder-id", CLEANED_DATA_FOLDER_ID.strip()])
if TEMPLATES_FOLDER_ID.strip():
    cmd.extend(["--templates-folder-id", TEMPLATES_FOLDER_ID.strip()])
if DATA_MAP_DESTINATION.strip():
    cmd.extend(["--data-map-destination", DATA_MAP_DESTINATION.strip()])
cmd.extend(["--existing-file-policy", EXISTING_FILE_POLICY])

display_cmd = ["<ACCESS_TOKEN>" if part == access_token else part for part in cmd]
print("Running:")
print(" ".join(display_cmd))

completed = subprocess.run(cmd, text=True, capture_output=True)
print("STDOUT")
print(completed.stdout)
print("STDERR")
print(completed.stderr)
completed.check_returncode()

In [ ]:
# @title 7. Show Latest Local Workflow Log {"display-mode":"form"}
from pathlib import Path

histories = study_folder / "histories"
if not histories.exists():
    print("No histories folder found.")
else:
    latest_history = sorted(path for path in histories.iterdir() if path.is_dir())[-1]
    log_path = latest_history / "log.md"
    print("Latest history:", latest_history)
    if log_path.exists():
        print("Log path:", log_path)
        print(log_path.read_text(encoding="utf-8")[-8000:])
    else:
        print("No log.md found in latest history folder.")

In [ ]:
# @title 8. Optional: Download Updated Local Logs {"display-mode":"form"}
from google.colab import files
import shutil

DOWNLOAD_LOGS = False  # @param {type:"boolean"}

if DOWNLOAD_LOGS:
    archive_base = Path("/content") / f"{study_folder.name}-histories"
    archive_path = shutil.make_archive(str(archive_base), "zip", study_folder / "histories")
    print("Downloading", archive_path)
    files.download(archive_path)
else:
    print("Set DOWNLOAD_LOGS=True and rerun this cell to download local workflow logs.")